In [3]:
import torch
import cv2
import numpy as np
from ultralytics import YOLO
from torchvision import transforms, models
import torch.nn as nn
from PIL import Image
import os
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd

# -------------------------
# Setup
# -------------------------
IMAGE_PATH = r"C:\project\dataset\seeds\images\train\pepper 215.jpg"
OUTPUT_DIR = "outputs_steps"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/crops", exist_ok=True)

# -------------------------
# Load YOLO Model
# -------------------------
yolo = YOLO(r"C:\project\seed_yolovm\weights\best.pt")

# -------------------------
# Load MobileNet Classifier
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CLASSES = ["others", "papaya", "pepper"]

model = models.mobilenet_v2()
model.classifier[1] = nn.Linear(model.last_channel, len(CLASSES))
model.load_state_dict(torch.load("C:\project\mobilenetv2_best.pth", map_location=DEVICE))
model.to(DEVICE)
model.eval()

tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# -------------------------
# STEP 1: Save Original Image
# -------------------------
img = cv2.imread(IMAGE_PATH)
cv2.imwrite(f"{OUTPUT_DIR}/step1_original.jpg", img)

# -------------------------
# STEP 2: YOLO Detection
# -------------------------
# STEP 2: YOLO Detection (Improved for dense seeds)
results = yolo(
    img,
    conf=0.10,   # Lower confidence → detects MANY more seeds
    iou=0.30,    # Lower IoU → keeps overlapping boxes (dense seeds)
    max_det=500  # Allow up to 500 seeds in one image
)[0]

yolo_img = results.plot()
cv2.imwrite(f"{OUTPUT_DIR}/step2_yolo_detection.jpg", yolo_img)

# -------------------------
# STEP 3: Crop YOLO Boxes
# -------------------------
crop_info = []
for i, box in enumerate(results.boxes):
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    conf_yolo = float(box.conf[0])

    crop = img[y1:y2, x1:x2]
    crop_path = f"{OUTPUT_DIR}/crops/seed_{i+1}.jpg"
    cv2.imwrite(crop_path, crop)

    crop_info.append((i+1, crop_path, (x1, y1, x2, y2), conf_yolo))

# -------------------------
# STEP 4: Classification + Confidence
# -------------------------
class_counts = {"pepper": 0, "papaya": 0, "others": 0}
results_list = []

for seed_id, path, coords, yolo_conf in crop_info:
    crop_img = Image.open(path).convert("RGB")
    crop_tensor = tfm(crop_img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(crop_tensor)
        probs = F.softmax(logits, dim=1)
        pred = probs.argmax(1).item()

    cls_label = CLASSES[pred]
    cls_conf = float(probs[0][pred])
    class_counts[cls_label] += 1

    results_list.append({
        "Seed_ID": seed_id,
        "Class": cls_label,
        "Class_Conf": cls_conf,
        "YOLO_Conf": yolo_conf,
        "x1": coords[0],
        "y1": coords[1],
        "x2": coords[2],
        "y2": coords[3]
    })

# Write summary to text
with open(f"{OUTPUT_DIR}/step4_classification.txt", "w",encoding="utf-8") as f:
    for r in results_list:
        f.write(f"Seed {r['Seed_ID']} → {r['Class']} (MobileNet={r['Class_Conf']:.2f}, YOLO={r['YOLO_Conf']:.2f})\n")

    f.write("\nCLASS COUNTS:\n")
    for c in class_counts:
        f.write(f"{c}: {class_counts[c]}\n")

    f.write(f"\nTotal Seeds: {len(results_list)}\n")

# -------------------------
# STEP 5: Annotated Final Image
# -------------------------
final_img = img.copy()

for r in results_list:
    x1, y1, x2, y2 = r["x1"], r["y1"], r["x2"], r["y2"]
    label = f"{r['Class']} {r['Class_Conf']:.2f}"
    cv2.rectangle(final_img, (x1, y1), (x2, y2), (0,255,0), 2)
    cv2.putText(final_img, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 
                0.6, (255,0,0), 2)

cv2.imwrite(f"{OUTPUT_DIR}/step5_final_output.jpg", final_img)

# -------------------------
# STEP 6: PIE CHART (Class Distribution)
# -------------------------
labels = class_counts.keys()
sizes = class_counts.values()

plt.figure(figsize=(6,6))
plt.pie(sizes, labels=labels, autopct="%1.1f%%")
plt.title("Class Distribution of Seeds")
plt.savefig(f"{OUTPUT_DIR}/step6_class_distribution.png")
plt.close()

# -------------------------
# STEP 7: BAR CHART (Confidence Scores)
# -------------------------
seed_ids = [r["Seed_ID"] for r in results_list]
conf_scores = [r["Class_Conf"] for r in results_list]

plt.figure(figsize=(10,5))
plt.bar(seed_ids, conf_scores)
plt.xlabel("Seed ID")
plt.ylabel("MobileNet Confidence")
plt.title("Confidence per Classified Seed")
plt.savefig(f"{OUTPUT_DIR}/step7_confidence_bars.png")
plt.close()

# -------------------------
# STEP 8: CSV EXPORT
# -------------------------
df = pd.DataFrame(results_list)
df.to_csv(f"{OUTPUT_DIR}/step8_results.csv", index=False)

print("\n🎉 ALL OUTPUTS GENERATED!")
print("Check folder:", OUTPUT_DIR)


C:\Users\sushm\AppData\Local\Temp\ipykernel_6484\3271319192.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("C:\project\mobilenetv2_bes


0: 640x640 7 seeds, 47.3ms
Speed: 4.3ms preprocess, 47.3ms inference, 3.4ms postprocess per image at shape (1, 3, 640, 640)

🎉 ALL OUTPUTS GENERATED!
Check folder: outputs_steps


In [10]:
import torch
import cv2
import numpy as np
from ultralytics import YOLO
from torchvision import transforms, models
import torch.nn as nn
from PIL import Image
import os
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd

# ==========================================
# CONFIG
# ==========================================
IMAGE_PATH = r"C:\Users\sushm\OneDrive\Desktop\mixed_012.jpg"
OUTPUT_DIR = "final_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/crops", exist_ok=True)

YOLO_MODEL = r"C:\project\seed_yolovm\weights\best.pt"
MOBILENET_MODEL = r"C:\project\mobilenetv3_best.pth"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CLASSES = ["others", "papaya", "pepper"]

# ==========================================
# SAFE IMAGE LOADER (same as your dataset code)
# ==========================================
def safe_imread(path):
    try:
        pil_img = Image.open(path).convert("RGB")
        return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    except:
        print(f"❌ Error reading: {path}")
        return None

# ==========================================
# LOAD YOLO
# ==========================================
yolo = YOLO(YOLO_MODEL)

# ==========================================
# LOAD MobileNetV3-small
# ==========================================
model = models.mobilenet_v3_small(weights=None)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(CLASSES))
model.load_state_dict(torch.load(MOBILENET_MODEL, map_location=DEVICE))

model.to(DEVICE)
model.eval()

tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ==========================================
# STEP 1: ORIGINAL IMAGE
# ==========================================
img = safe_imread(IMAGE_PATH)
cv2.imwrite(f"{OUTPUT_DIR}/step1_original.jpg", img)

# ==========================================
# STEP 2: YOLO DETECTION (use dataset logic)
# ==========================================
results = yolo(img, conf=0.50, iou=0.40)[0]
yolo_det_img = results.plot()
cv2.imwrite(f"{OUTPUT_DIR}/step2_yolo_detection.jpg", yolo_det_img)

# ==========================================
# STEP 3: CROP DETECTED SEEDS (dataset-style)
# ==========================================
crop_info = []
crop_id = 0

for box in results.boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0])

    if x2 <= x1 or y2 <= y1:
        continue

    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        continue

    crop_id += 1
    crop_path = f"{OUTPUT_DIR}/crops/crop_{crop_id}.jpg"
    cv2.imwrite(crop_path, crop)

    crop_info.append((crop_id, crop_path, (x1, y1, x2, y2), float(box.conf[0])))

# ==========================================
# STEP 4: CLASSIFICATION
# ==========================================
results_list = []
class_counts = {"others": 0, "papaya": 0, "pepper": 0}

for seed_id, crop_path, coords, yolo_conf in crop_info:

    crop_img = Image.open(crop_path).convert("RGB")
    crop_tensor = tfm(crop_img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(crop_tensor)
        probs = F.softmax(logits, dim=1)[0]

    pred = torch.argmax(probs).item()
    cls_label = CLASSES[pred]
    cls_conf = float(probs[pred])

    class_counts[cls_label] += 1

    results_list.append({
        "Seed_ID": seed_id,
        "Class": cls_label,
        "Class_Conf": cls_conf,
        "YOLO_Conf": yolo_conf,
        "x1": coords[0],
        "y1": coords[1],
        "x2": coords[2],
        "y2": coords[3]
    })

# SAVE TXT SUMMARY
with open(f"{OUTPUT_DIR}/step4_classification.txt", "w", encoding="utf-8") as f:
    for r in results_list:
        f.write(f"Seed {r['Seed_ID']} → {r['Class']} (MobileNet={r['Class_Conf']:.2f}, YOLO={r['YOLO_Conf']:.2f})\n")

    f.write("\nCLASS COUNTS:\n")
    for c in class_counts:
        f.write(f"{c}: {class_counts[c]}\n")

    f.write(f"\nTOTAL SEEDS: {len(results_list)}\n")

# ==========================================
# STEP 5: FINAL ANNOTATED IMAGE
# ==========================================
final_img = img.copy()

for r in results_list:
    x1, y1, x2, y2 = r["x1"], r["y1"], r["x2"], r["y2"]
    label = f"{r['Class']} {r['Class_Conf']:.2f}"

    cv2.rectangle(final_img, (x1, y1), (x2, y2), (0,255,0), 2)
    cv2.putText(final_img, label, (x1, y1 - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

cv2.imwrite(f"{OUTPUT_DIR}/step5_final_output.jpg", final_img)

# ==========================================
# STEP 6: PIE CHART
# ==========================================
labels = class_counts.keys()
sizes = class_counts.values()

plt.figure(figsize=(6,6))
plt.pie(sizes, labels=labels, autopct='%1.1f%%')
plt.title("Class Distribution")
plt.savefig(f"{OUTPUT_DIR}/step6_pie_chart.png")
plt.close()

# ==========================================
# STEP 7: CONFIDENCE BAR CHART
# ==========================================
seed_ids = [r["Seed_ID"] for r in results_list]
conf_scores = [r["Class_Conf"] for r in results_list]

plt.figure(figsize=(10,5))
plt.bar(seed_ids, conf_scores)
plt.xlabel("Seed ID")
plt.ylabel("MobileNet Confidence")
plt.title("Confidence per Seed")
plt.savefig(f"{OUTPUT_DIR}/step7_confidence_bars.png")
plt.close()

# ==========================================
# STEP 8: CSV EXPORT
# ==========================================
df = pd.DataFrame(results_list)
df.to_csv(f"{OUTPUT_DIR}/results.csv", index=False)

print("\n🎉 ALL 7 OUTPUTS GENERATED SUCCESSFULLY!")
print("📁 OUTPUT FOLDER:", OUTPUT_DIR)


C:\Users\sushm\AppData\Local\Temp\ipykernel_24820\57431255.py:48: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MOBILENET_MODEL, map_locatio


0: 640x480 177 seeds, 38.7ms
Speed: 4.9ms preprocess, 38.7ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 480)

🎉 ALL 7 OUTPUTS GENERATED SUCCESSFULLY!
📁 OUTPUT FOLDER: final_outputs
